<a href="https://colab.research.google.com/github/sachacn02/Robust_Opt/blob/main/src/robust_opt/Simple_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ROBUSTIFYING A SIMPLE LINEAR PROGRAM EXAMPLE

Modified by:
 1. Erick Delage (Created for ROME 14 April 2015)
 2. Erick Delage (Adapted to RSOME in November 2020)

As discussed in example 2.5 of the  [lecture notes](http://tintin.hec.ca/pages/erick.delage/MATH80624_LectureNotes.pdf) of MATH80624 at HEC Montréal.

WARNING!!!

The following code exploits a free Mosek licence for the course "MATH80624A" offered at HEC Montréal (expiration December 31st 2025). If you have error messages informing you about licencing issues, you may try uncommenting the installation lines for Gurobi. Otherwise, we recommend that you obtain your own licence of either Mosek ([url](https://www.mosek.com/)) or Gurobi ([url](https://www.gurobi.com/)).

# **Preliminaries**

In [ ]:
!pip install rsome
!pip install mosek
!rm mosek.lic
!git clone https://github.com/erickdelage/80624
!cp ./80624/mosek.lic .
!rm -r ./80624
!mkdir -p /root/mosek
!cp ./mosek.lic /root/mosek
#!pip install -i https://pypi.gurobi.com gurobipy


Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 62 kB 800 kB/s 
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 10.1 MB 35.4 MB/s 
rm: cannot remove 'mosek.lic': No such file or directory
Cloning into '80624'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 18 (delta 3), reused 17 (delta 2), pack-reused 0
Unpacking objects: 100% (18/18), done.


In [ ]:
import rsome as rso
import numpy as np
from rsome import ro
from rsome import msk_solver as my_solver  #Import Mosek solver interface
#from rsome import grb_solver as my_solver  #Import Gurobi solver interface


# **Simple Example with Box uncertainty**

In [ ]:
# Parameters setup
n  = 150
i = np.arange(1, n+1)
c  = 0.15+ i*0.05/150
a = np.zeros(n)
b = 0.02

# Describing the uncertainty
maxDev = 0.05/450 * (2*i*n*(n+1))**0.5
zBarPlus = maxDev
zBarMinus = -maxDev

## **Solving Deterministic Model**

Consider the following linear programming problem:
\begin{align}
\max\limits_{x}\;\;&c^\top x \\
\text{subject to}\;\;& a^\top x \leq b\\
& 0 \leq x \leq 1.
\end{align}


In [ ]:

#Create model
model = ro.Model('simpleExample_det')
x=model.dvar(n)
model.max(c@x)
model.st(a@x<=b)
model.st(x>=0)
model.st(x<=1)

model.solve(my_solver)
optobj_det = model.get()
xx_det   = x.get()
print('objective',optobj_det)
print('solution',xx_det)



Being solved by Mosek...
Solution status: optimal
Running time: 0.0139s
objective 26.275
solution [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1.]


## **Solving Raw Robust Counterpart**

Consider the following robust linear optimization problem:
\begin{align}
\max\limits_{x}\;\;&c^\top x \\
\text{subject to}\;\;& (a + z)^\top x \leq b && \forall z \in \mathcal{Z}\\
& 0 \leq x \leq 1,
\end{align}
where $\mathcal{Z}:=\{z\;|\;\bar{z}^- \leq z \leq \bar{z}^+\}$.

In [ ]:
#Create model
model = ro.Model('simpleExample_rawrobust')
x=model.dvar(n)

#Create uncertain vector
z= model.rvar(n)
#Create uncertainty set
boxSet= (z>=zBarMinus, z <= zBarPlus)

model.max(c@x)
#Robustify the constraint
model.st(((a+z)@x<=b).forall(boxSet))
model.st(x>=0)
model.st(x<=1)
model.solve(my_solver)

optobj_raw = model.get()
xx_raw   = x.get()

print('objective',optobj_raw)
print('solution',xx_raw)

Being solved by Mosek...
Solution status: optimal
Running time: 0.0090s
objective 0.12713897073987465
solution [ 0.84571377 -0.         -0.         -0.         -0.         -0.
 -0.         -0.         -0.         -0.         -0.         -0.
 -0.         -0.         -0.         -0.         -0.         -0.
 -0.         -0.         -0.         -0.         -0.         -0.
 -0.         -0.         -0.         -0.         -0.         -0.
 -0.         -0.         -0.         -0.         -0.         -0.
 -0.         -0.         -0.         -0.         -0.         -0.
 -0.         -0.         -0.         -0.         -0.         -0.
 -0.         -0.         -0.         -0.         -0.         -0.
 -0.         -0.         -0.         -0.         -0.         -0.
 -0.         -0.         -0.         -0.         -0.         -0.
 -0.         -0.         -0.         -0.         -0.         -0.
 -0.         -0.         -0.         -0.         -0.         -0.
 -0.         -0.         -0.         -0.    

## **Solving Reduced Robust Counterpart**

The robust counterpart can be reformulated as follows:

\begin{align}
\max\limits_{x,\lambda^+,\lambda^-}\;\;&c^\top x \\
\text{subject to}\;\;& a ^\top x + (\bar{z}^+)^\top \lambda^+ - (\bar{z}^-)^\top \lambda^- \leq b &&\\
& \lambda^+ -\lambda^- =x \\
& \lambda^- \geq 0, \lambda^+ \geq 0\\
& 0 \leq x \leq 1.
\end{align}

In [ ]:
#Create model
model = ro.Model('simpleExample_redrobust')
x=model.dvar(n)
#Create auxiliary variables
lambdaPlus=model.dvar(n)
lambdaMinus=model.dvar(n)

model.max(c@x)
#Modify the deterministic constraint
model.st(a@x + zBarPlus@lambdaPlus -zBarMinus@lambdaMinus <=b)
#Add constraints from dual representation of worst-case optimization
model.st(lambdaPlus-lambdaMinus == x)
model.st(lambdaPlus>=0)
model.st(lambdaMinus>=0)

model.st(x>= 0)
model.st(x<=1)

model.solve(my_solver)

optobj_red = model.get()
xx_red   = x.get()

print('objective',optobj_red)
print('solution',xx_red)

Being solved by Mosek...
Solution status: optimal
Running time: 0.0158s
objective 0.12713897073987465
solution [0.84571377 0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0. 